# Annotation Agent — EDA & Quality Report

This notebook covers:
1. Load labeled data
2. Label distribution
3. Confidence distribution
4. Text length analysis
5. Top-20 words per class
6. Flagged (low-confidence) examples
7. Cohen's κ vs human labels (after human annotation)
8. Strategy justification


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
import re, json
from pathlib import Path

matplotlib.rcParams['figure.figsize'] = (10, 4)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/labeled.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#4CAF50', '#F44336', '#2196F3'])
axes[0].set_title('Label Counts')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 0.3, str(v), ha='center')

# Pie
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336', '#2196F3'])
axes[1].set_title('Label Distribution')

plt.tight_layout()
plt.savefig('../data/label_distribution.png', dpi=120)
plt.show()
print(counts)

## 2. Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['confidence'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(0.75, color='red', linestyle='--', label='threshold=0.75')
axes[0].set_title('Confidence Distribution')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].legend()

# By label
for lbl in df['label'].dropna().unique():
    axes[1].hist(df[df['label']==lbl]['confidence'], bins=15, alpha=0.6, label=lbl)
axes[1].axvline(0.75, color='red', linestyle='--', label='threshold')
axes[1].set_title('Confidence by Label')
axes[1].set_xlabel('Confidence')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/confidence_distribution.png', dpi=120)
plt.show()
print(f"Mean confidence: {df['confidence'].mean():.3f}")
print(f"Flagged rows: {df['flagged_for_review'].sum()} ({df['flagged_for_review'].mean()*100:.1f}%)")

## 3. Text Length Distribution

In [ ]:
df['text_len'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['text_len'], bins=20, color='mediumpurple', edgecolor='white')
axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')

axes[1].hist(df['word_count'], bins=20, color='darkorange', edgecolor='white')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')

plt.tight_layout()
plt.savefig('../data/text_length.png', dpi=120)
plt.show()
df[['text_len', 'word_count']].describe()

## 4. Top-20 Words per Class

In [ ]:
STOPWORDS = {'the','a','an','and','or','but','in','on','at','to','for',
             'of','with','it','is','was','i','my','this','that','be','not'}

def top_words(texts, n=20):
    words = []
    for t in texts:
        words += [w.lower() for w in re.findall(r'\b[a-zA-Z]+\b', str(t)) if w.lower() not in STOPWORDS]
    return Counter(words).most_common(n)

labels = df['label'].dropna().unique()
fig, axes = plt.subplots(1, len(labels), figsize=(6*len(labels), 5))
if len(labels) == 1:
    axes = [axes]

for ax, lbl in zip(axes, labels):
    words, counts = zip(*top_words(df[df['label']==lbl]['text']))
    ax.barh(list(words)[::-1], list(counts)[::-1])
    ax.set_title(f'Top words: {lbl}')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../data/top_words.png', dpi=120)
plt.show()

## 5. Flagged Examples (Low Confidence)

In [ ]:
flagged = df[df['flagged_for_review']].sort_values('confidence')
print(f'Total flagged: {len(flagged)}')
flagged[['text', 'label', 'confidence']].head(15)

## 6. Cohen's κ vs Human Labels

Run this cell after a human annotator has labeled the data and you've added a `human_label` column.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

if 'human_label' in df.columns:
    mask = df['label'].notna() & df['human_label'].notna()
    kappa = cohen_kappa_score(df.loc[mask, 'human_label'], df.loc[mask, 'label'])
    print(f'Cohen\'s κ (model vs human): {kappa:.4f}')

    # Confusion matrix
    labels_order = sorted(df['label'].dropna().unique())
    cm = confusion_matrix(df.loc[mask, 'human_label'], df.loc[mask, 'label'], labels=labels_order)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels_order, yticklabels=labels_order,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Model label')
    ax.set_ylabel('Human label')
    ax.set_title(f"Model vs Human — Cohen's κ = {kappa:.3f}")
    plt.tight_layout()
    plt.savefig('../data/kappa_confusion.png', dpi=120)
    plt.show()
else:
    print('No human_label column yet. Add human annotations and re-run.')
    print('To add: merge low_confidence.csv annotations back into labeled.csv as human_label column.')

## 7. Quality Metrics Summary

In [ ]:
metrics_path = Path('../data/quality_metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([
        {'Metric': 'Total rows', 'Value': len(df)},
        {'Metric': 'Confidence mean', 'Value': metrics.get('confidence_mean')},
        {'Metric': 'Confidence std', 'Value': metrics.get('confidence_std')},
        {'Metric': 'Flagged count', 'Value': metrics.get('flagged_count')},
        {'Metric': 'Flagged %', 'Value': f"{metrics.get('flagged_pct')}%"},
        {'Metric': "Cohen's κ", 'Value': metrics.get('kappa', 'N/A — need human labels')},
    ]))

## 8. Strategy Justification

### Why zero-shot classification for sentiment?

**Model:** `facebook/bart-large-mnli` (Natural Language Inference)

**Why this approach:**
- Zero-shot works without any labeled training examples — ideal when starting a new annotation project
- NLI framing ("does this text express positive sentiment?") aligns naturally with sentiment classification
- Produces calibrated probability scores → confidence thresholding is meaningful

**Human-in-the-loop threshold = 0.75:**
- Examples below this threshold are genuinely ambiguous (sarcasm, mixed sentiment, very short texts)
- Sending only ~20-30% of data for human review is more cost-effective than full manual annotation
- After human review, Cohen's κ measures how well the model agrees with humans on the confident predictions

**Limitations:**
- Model may struggle with domain-specific language (e.g. technical reviews)
- Sarcasm is reliably misclassified → always flag short texts for human review
- For production: fine-tune on ~500 human-labeled examples to boost accuracy significantly

**Expected Cohen's κ:** 0.65-0.80 on confident predictions (literature baseline for NLI-based zero-shot)
